# NetForensics AI: An Intelligent Network Fault Detection and Troubleshooting System

## Phase 3: AI-Assisted Network Fault Diagnosis

### Project Overview

NetForensics AI combines rule-based troubleshooting with AI-assisted network fault diagnosis.

In this phase, structured information from the network troubleshooting dataset is used to create AI prompts. The AI system receives information about the network problem and generates a structured diagnosis.

The diagnosis includes the probable root cause, confidence level, supporting evidence, OSI layer, recommended next command, and suggested fix steps.

### Objectives

The objectives of this notebook are:

- Load the network troubleshooting dataset.
- Select relevant information for AI diagnosis.
- Create structured prompts for network fault analysis.
- Define a standard diagnosis output format.
- Generate AI-ready prompts for troubleshooting cases.
- Prepare the dataset for AI-assisted diagnosis.
- Save the generated prompts and diagnosis results.

This phase provides the AI-assisted diagnostic component of the NetForensics AI system.

## 1. Importing Required Libraries

The Pandas library is used to load and process the network troubleshooting dataset.

In [1]:
import pandas as pd

## 2. Loading the Network Fault Dataset

The validated network troubleshooting dataset is loaded for AI-assisted diagnosis.

The dataset contains network fault cases along with symptoms, evidence, topology information, protocols, expected faults, and troubleshooting details.

In [2]:
df = pd.read_csv("cases.csv")

print("Dataset loaded successfully.")
print("Number of cases:", len(df))

Dataset loaded successfully.
Number of cases: 100


## 3. Displaying Available Dataset Columns

This step displays the available dataset columns.

The columns will be used to provide structured information to the AI diagnosis system.

In [3]:
print("Dataset Columns:")

for column in df.columns:
    print(column)

Dataset Columns:
Case_ID
Category
Difficulty
Topology
Symptom
Evidence
Expected_Fault
OSI_Layer
Concept
Severity
Expected_Next_Command
Expected_Fix
Verification 
Notes


## 4. AI Diagnosis Input

For each network fault case, the AI system requires relevant troubleshooting information.

The main inputs used for diagnosis are:

- Network topology
- Protocol
- Symptom
- Evidence
- OSI layer

These inputs provide context about the network problem and help the AI generate a structured diagnosis.

## 5. Define the AI Diagnosis Prompt

A structured prompt is created to ensure that the AI analyzes each network fault case in a consistent manner.

The AI is instructed to use only the information provided in the network case and avoid unsupported assumptions.

In [4]:
def create_prompt(topology, protocol, symptom, evidence):

    prompt = f"""
You are a network troubleshooting assistant.

Analyze the following network fault case using only the information provided.

Network Topology:
{topology}

Protocol or Technology:
{protocol}

Symptom:
{symptom}

Evidence:
{evidence}

Provide the diagnosis in the following format:

Root Cause:
Confidence:
Evidence Used:
OSI Layer:
Recommended Next Command:
Suggested Fix Steps:

Do not make unsupported assumptions.
If the available evidence is insufficient, mention that additional troubleshooting is required.
"""

    return prompt

## 6. Generate a Sample AI Prompt

Before generating prompts for all cases, a sample case is used to verify the structure of the AI prompt.

In [5]:
sample = df.iloc[0]

sample_prompt = create_prompt(
    sample["Topology"],
    sample["Category"],
    sample["Symptom"],
    sample["Evidence"]
)

print(sample_prompt)


You are a network troubleshooting assistant.

Analyze the following network fault case using only the information provided.

Network Topology:
PC1->Switch1->Router1->Server1

Protocol or Technology:
VLAN

Symptom:
PC cannot reach gateway or server

Evidence:
show vlan brief, show interfaces fa0/1 switchport, ipconfig

Provide the diagnosis in the following format:

Root Cause:
Confidence:
Evidence Used:
OSI Layer:
Recommended Next Command:
Suggested Fix Steps:

Do not make unsupported assumptions.
If the available evidence is insufficient, mention that additional troubleshooting is required.



## 7. Generate AI Prompts for All Network Cases

The structured prompt function is now applied to every network troubleshooting case.

Each case receives an individual AI prompt containing its topology, protocol, symptom, and evidence.

In [7]:
prompts = []

for index, row in df.iterrows():

    prompt = create_prompt(
        row["Topology"],
        row["Category"],
        row["Symptom"],
        row["Evidence"]
    )

    prompts.append(prompt)

print("Total AI prompts created:", len(prompts))

Total AI prompts created: 100


## 8. Add AI Prompts to the Dataset

The generated prompts are stored in a new column.

This allows each network troubleshooting case to be directly connected with its AI diagnosis input.

In [8]:
df["AI Prompt"] = prompts

print("AI prompts added to the dataset successfully.")

AI prompts added to the dataset successfully.


## 9. Display Generated AI Prompts

This step displays selected cases along with their generated AI prompts.

It helps verify that the required troubleshooting information is correctly included in the prompt.

In [9]:
df[[
    "Case_ID",
    "Category",
    "Symptom",
    "AI Prompt"
]].head(3)

,Case_ID,Category,Symptom,AI Prompt
0,C001,VLAN,PC cannot reach gateway or server,\nYou are a network troubleshooting assistant....
1,C002,VLAN,PC cannot reach gateway/server,\nYou are a network troubleshooting assistant....
2,C003,VLAN/Trunk,PC1 cannot reach PC2,\nYou are a network troubleshooting assistant....


## 10. Define the Expected AI Diagnosis Structure

For consistency, every AI diagnosis should follow the same structured format.

The AI output should contain the following information:

- Root Cause
- Confidence
- Evidence Used
- OSI Layer
- Recommended Next Command
- Suggested Fix Steps

This structured format makes the AI response easier to analyze and compare with the expected troubleshooting results.

In [10]:
diagnosis_fields = [
    "Root Cause",
    "Confidence",
    "Evidence Used",
    "OSI_Layer",
    "Recommended Next Command",
    "Suggested Fix Steps"
]

print("Expected AI Diagnosis Fields:")

for field in diagnosis_fields:
    print("-", field)

Expected AI Diagnosis Fields:
- Root Cause
- Confidence
- Evidence Used
- OSI_Layer
- Recommended Next Command
- Suggested Fix Steps


## 11. Example of Structured AI Diagnosis

The following example shows the expected format of an AI-generated network fault diagnosis.

In [11]:
example_diagnosis = {
    "Root Cause": "Possible network configuration issue",
    "Confidence": "Medium",
    "Evidence Used": "Network connectivity failure observed",
    "OSI Layer": "Network Layer",
    "Recommended Next Command": "show ip interface brief",
    "Suggested Fix Steps": "Verify interface status and IP configuration"
}

for key, value in example_diagnosis.items():
    print(key + ":", value)

Root Cause: Possible network configuration issue
Confidence: Medium
Evidence Used: Network connectivity failure observed
OSI Layer: Network Layer
Recommended Next Command: show ip interface brief
Suggested Fix Steps: Verify interface status and IP configuration


## 12. Install and Import the AI Model Library

The Hugging Face Transformers library is used to access a pre-trained language model.

The model will be used to generate AI-assisted responses based on the structured network troubleshooting prompts.

In [15]:
!pip install transformers torch


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 13. Load a Pre-trained Language Model

A pre-trained language model is loaded using the Hugging Face pipeline.

The model receives a network troubleshooting prompt and generates a text-based response.

A lightweight model is selected so that it can be tested on a standard computer.

In [17]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

print("AI model loaded successfully.")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 4137.63it/s]


AI model loaded successfully.


## 14. Test AI-Assisted Diagnosis

Before applying the AI model to multiple network cases, a single generated prompt is tested.

This step verifies whether the AI model can generate a response based on the network troubleshooting information.

In [19]:
result = generator(
    sample_prompt,
    max_new_tokens=100,
    num_return_sequences=1
)

print(result[0]["generated_text"])

[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a network troubleshooting assistant.

Analyze the following network fault case using only the information provided.

Network Topology:
PC1->Switch1->Router1->Server1

Protocol or Technology:
VLAN

Symptom:
PC cannot reach gateway or server

Evidence:
show vlan brief, show interfaces fa0/1 switchport, ipconfig

Provide the diagnosis in the following format:

Root Cause:
Confidence:
Evidence Used:
OSI Layer:
Recommended Next Command:
Suggested Fix Steps:

Do not make unsupported assumptions.
If the available evidence is insufficient, mention that additional troubleshooting is required.
Remove the network configuration from the network and use the following command:
root@localhost
Remove network configuration from the network and use the following command:
root@localhost
Remove network configuration from the network and use the following command:
root@localhost
Remove network configuration from the network and use the following command:
root@localhost
Remove network configuration

## 15. Initial AI Model Evaluation

The initial test was performed using the DistilGPT-2 language model.

The model successfully generated text based on the network troubleshooting prompt. However, the generated response did not consistently follow the required diagnosis format and contained irrelevant information.

This occurred because DistilGPT-2 is a general-purpose text generation model and is not specifically designed for instruction-based network troubleshooting.

Therefore, the initial model is not considered sufficiently reliable for structured network fault diagnosis. A more suitable AI-assisted approach will be used for the next stage of the NetForensics AI system.

## 16. Improved AI Model Selection

The initial DistilGPT-2 model produced unstructured and irrelevant responses because it is a general-purpose text generation model.

To improve instruction following, the NetForensics AI system uses the FLAN-T5 Small model. FLAN-T5 is an instruction-tuned language model that is more suitable for generating responses based on structured prompts.

The model is used to generate AI-assisted network fault diagnoses from the available symptom and evidence.

In [23]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("FLAN-T5 model loaded successfully.")

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 2830.56it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


FLAN-T5 model loaded successfully.


## 17. Create an Improved Network Diagnosis Prompt

The prompt is simplified to make it easier for the AI model to understand the network problem.

The model is instructed to provide a concise diagnosis based only on the available information.

In [32]:
def create_diagnosis_prompt(protocol, symptom, evidence):

    prompt = f"""
Network troubleshooting task.

Protocol: {protocol}
Problem: {symptom}
Evidence: {evidence}

What is the most likely network fault? Give a short answer.
"""

    return prompt

## 18. Test the Improved AI Model

A sample network fault case is sent to the FLAN-T5 model to test whether the generated diagnosis is more relevant and structured.

In [34]:
sample = df.iloc[0]

test_prompt = create_diagnosis_prompt(
    sample["Category"],
    sample["Symptom"],
    sample["Evidence"]
)

print(test_prompt)


Network troubleshooting task.

Protocol: VLAN
Problem: PC cannot reach gateway or server
Evidence: show vlan brief, show interfaces fa0/1 switchport, ipconfig

What is the most likely network fault? Give a short answer.



In [36]:
inputs = tokenizer(
    test_prompt,
    return_tensors="pt",
    truncation=True
)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False
)

ai_response = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("AI Diagnosis:", ai_response)

AI Diagnosis: ipconfig


## 19. AI Model Test Observation

The FLAN-T5 Small model was tested using network troubleshooting cases.

The model was able to generate a response; however, some responses were incomplete or returned a troubleshooting command instead of identifying the actual network fault.

For example, the model returned "ipconfig", which is a diagnostic command rather than a root cause.

This experiment shows that a small general-purpose language model may not provide sufficiently reliable network fault diagnosis without additional domain knowledge or stronger instruction capabilities.

Therefore, the AI-assisted component should be combined with the structured knowledge available in the NetForensics AI dataset rather than relying only on raw language model generation.

## 20. Hybrid AI-Assisted Diagnosis Approach

The initial language model experiments showed that small general-purpose language models may generate incomplete or unreliable diagnoses for network troubleshooting cases.

Therefore, NetForensics AI uses a hybrid approach based on dataset knowledge and text similarity.

The system compares the symptom and evidence of a network problem with existing network fault cases.

The most similar case is identified using TF-IDF vectorization and cosine similarity.

This approach helps provide a diagnosis based on previously documented network troubleshooting knowledge.

## 21. Import Required Machine Learning Libraries

Scikit-learn is used for text vectorization and similarity calculation.

TF-IDF converts textual information into numerical form, while cosine similarity is used to identify similar network fault cases.

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 22. Prepare Text Data for Similarity Analysis

The symptom and evidence columns are combined to create a single text representation for each network troubleshooting case.

This combined information will be used to identify similar network problems.

In [39]:
df["Case Text"] = (
    df["Symptom"].fillna("") + " " +
    df["Evidence"].fillna("")
)

df[["Case_ID", "Case Text"]].head()

,Case_ID,Case Text
0,C001,PC cannot reach gateway or server show vlan br...
1,C002,PC cannot reach gateway/server VLAN and switch...
2,C003,PC1 cannot reach PC2 show interfaces trunk on ...
3,C004,PC1 does not receive a vali 192.168.10.x addre...
4,C005,PC1 cannot reach default gateway show ip inter...


## 23. Convert Network Case Text into Numerical Features

TF-IDF vectorization is used to convert the textual network case information into numerical features.

These features allow the system to compare similarities between different network troubleshooting cases.

In [40]:
vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(df["Case Text"])

print("TF-IDF matrix created successfully.")
print("Matrix shape:", tfidf_matrix.shape)

TF-IDF matrix created successfully.
Matrix shape: (100, 603)


## 24. Create Similarity Matrix

Cosine similarity is calculated between all network troubleshooting cases.

A higher similarity score indicates that two cases contain more similar textual information.

In [41]:
similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity matrix created successfully.")
print("Matrix shape:", similarity_matrix.shape)

Similarity matrix created successfully.
Matrix shape: (100, 100)


## 25. Create Similar Case Detection Function

The following function accepts a symptom and evidence as input.

It converts the input into TF-IDF features and identifies the most similar network fault case from the dataset.

In [42]:
def find_similar_case(symptom, evidence):

    input_text = str(symptom) + " " + str(evidence)

    input_vector = vectorizer.transform([input_text])

    similarity_scores = cosine_similarity(
        input_vector,
        tfidf_matrix
    )

    similar_index = similarity_scores.argmax()

    similarity_score = similarity_scores[0][similar_index]

    return similar_index, similarity_score

## 26. Test Similar Case Detection

A sample network troubleshooting case is used to test the similarity-based diagnosis system.

The system identifies the most similar case and displays its similarity score.

In [44]:
sample = df.iloc[0]

similar_index, score = find_similar_case(
    sample["Symptom"],
    sample["Evidence"]
)

print("Input Case ID:", sample["Case_ID"])
print("Most Similar Case ID:", df.iloc[similar_index]["Case_ID"])
print("Similarity Score:", round(score, 2))

Input Case ID: C001
Most Similar Case ID: C001
Similarity Score: 1.0


## 27. Exclude the Same Case from Similarity Matching

When an existing case is tested against the complete dataset, it can match itself with a similarity score of 1.0.

To make the similarity-based diagnosis more realistic, the current case is excluded from the comparison.

The system then identifies another network troubleshooting case with the highest similarity.

In [51]:
def find_similar_case_excluding_self(index):

    scores = similarity_matrix[index].copy()

    # Exclude the same case
    scores[index] = -1

    similar_index = scores.argmax()
    similarity_score = scores[similar_index]

    return similar_index, similarity_score

## 28. Test Similarity Matching Without Self-Matching

This step tests the similarity-based system by finding the most similar case other than the original case itself.

In [53]:
test_index = 0

similar_index, score = find_similar_case_excluding_self(test_index)

print("Original Case_ID:", df.iloc[test_index]["Case_ID"])
print("Similar Case_ID:", df.iloc[similar_index]["Case_ID"])
print("Similarity Score:", round(score, 2))
print("Similar Case Fault:", df.iloc[similar_index]["Expected_Fault"])

Original Case_ID: C001
Similar Case_ID: C002
Similarity Score: 0.56
Similar Case Fault: VLAN 10 missing on Switch1


## 29. Create Similarity-Based AI Diagnosis

The most similar historical network fault case is used as a knowledge reference.

The diagnosis system retrieves relevant troubleshooting information from the similar case, including:

- Expected fault
- Recommended troubleshooting command
- Suggested fix

The similarity score is also used to indicate how closely the current case matches the reference case.

In [59]:
def get_ai_diagnosis(index):

    similar_index, score = find_similar_case_excluding_self(index)

    diagnosis = {
        "Similar Case ID": df.iloc[similar_index]["Case_ID"],
        "Similarity Score": round(score, 2),
        "Probable Fault": df.iloc[similar_index]["Expected_Fault"],
        "Recommended Command": df.iloc[similar_index]["Expected_Next_Command"],
        "Suggested Fix": df.iloc[similar_index]["Expected_Fix"]
    }

    return diagnosis

## 30. Test the AI-Assisted Diagnosis Function

The similarity-based diagnosis function is tested on a sample network case.

The system retrieves information from the most similar historical case and provides a probable diagnosis.

In [60]:
diagnosis = get_ai_diagnosis(0)

for key, value in diagnosis.items():
    print(key + ":", value)

Similar Case ID: C002
Similarity Score: 0.56
Probable Fault: VLAN 10 missing on Switch1
Recommended Command: show vlan brief
Suggested Fix: Recreate VLAN 10 and assign required ports


## 31. Confidence Level Based on Similarity Score

The similarity score is converted into a confidence level.

A higher similarity score indicates that the current network problem is more similar to an existing network fault case.

The confidence levels are:

- High: Similarity score 0.70 or above
- Medium: Similarity score between 0.40 and 0.69
- Low: Similarity score below 0.40

In [62]:
def get_confidence(score):

    if score >= 0.70:
        return "High"
    elif score >= 0.40:
        return "Medium"
    else:
        return "Low"

## 32. Add Confidence Level to AI Diagnosis

The confidence level is added to the diagnosis based on the similarity score.

This helps indicate how strongly the current network case matches the similar historical case.

In [63]:
def get_ai_diagnosis(index):

    similar_index, score = find_similar_case_excluding_self(index)

    confidence = get_confidence(score)

    diagnosis = {
        "Similar Case ID": df.iloc[similar_index]["Case_ID"],
        "Similarity Score": round(score, 2),
        "Confidence": confidence,
        "Probable Fault": df.iloc[similar_index]["Expected_Fault"],
        "Recommended Command": df.iloc[similar_index]["Expected_Next_Command"],
        "Suggested Fix": df.iloc[similar_index]["Expected_Fix"]
    }

    return diagnosis

## 33. Test AI-Assisted Diagnosis with Confidence Level

The improved diagnosis function is tested on a sample network case.

The result includes the similar historical case, similarity score, confidence level, probable fault, recommended command, and suggested fix.

In [65]:
diagnosis = get_ai_diagnosis(0)

for key, value in diagnosis.items():
    print(key + ":", str(value))

Similar Case ID: C002
Similarity Score: 0.56
Confidence: Medium
Probable Fault: VLAN 10 missing on Switch1
Recommended Command: show vlan brief
Suggested Fix: Recreate VLAN 10 and assign required ports


## 34. Generate AI-Assisted Diagnosis for All Network Cases

The similarity-based diagnosis system is applied to all network troubleshooting cases.

For each case, the system identifies the most similar historical case and generates:

- Similar Case ID
- Similarity Score
- Confidence Level
- Probable Fault
- Recommended Command
- Suggested Fix

In [66]:
similar_case_ids = []
similarity_scores = []
confidence_levels = []
probable_faults = []
recommended_commands = []
suggested_fixes = []

for index in range(len(df)):

    diagnosis = get_ai_diagnosis(index)

    similar_case_ids.append(diagnosis["Similar Case ID"])
    similarity_scores.append(diagnosis["Similarity Score"])
    confidence_levels.append(diagnosis["Confidence"])
    probable_faults.append(diagnosis["Probable Fault"])
    recommended_commands.append(diagnosis["Recommended Command"])
    suggested_fixes.append(diagnosis["Suggested Fix"])

print("AI-assisted diagnosis completed for all cases.")

AI-assisted diagnosis completed for all cases.


## 35. Add AI Diagnosis Results to the Dataset

The generated AI-assisted diagnosis results are added as new columns to the dataset.

This allows the original network fault information and AI-assisted results to be analyzed together.

In [67]:
df["AI_Similar_Case_ID"] = similar_case_ids
df["AI_Similarity_Score"] = similarity_scores
df["AI_Confidence"] = confidence_levels
df["AI_Probable_Fault"] = probable_faults
df["AI_Recommended_Command"] = recommended_commands
df["AI_Suggested_Fix"] = suggested_fixes

print("AI diagnosis results added successfully.")

AI diagnosis results added successfully.


## 36. Display AI-Assisted Diagnosis Results

This step displays selected network cases along with the results generated by the similarity-based AI-assisted diagnosis system.

In [68]:
df[[
    "Case_ID",
    "Expected_Fault",
    "AI_Similar_Case_ID",
    "AI_Similarity_Score",
    "AI_Confidence",
    "AI_Probable_Fault",
    "AI_Recommended_Command"
]].head(10)

,Case_ID,Expected_Fault,AI_Similar_Case_ID,AI_Similarity_Score,AI_Confidence,AI_Probable_Fault,AI_Recommended_Command
0,C001,PC switch port assigned to wrong VLAN,C002,0.56,Medium,VLAN 10 missing on Switch1,show vlan brief
1,C002,VLAN 10 missing on Switch1,C001,0.56,Medium,PC switch port assigned to wrong VLAN,show vlan brief
2,C003,VLAN 10 not allowed on trunk,C007,0.51,Medium,R1 has incorrect static route next hop 10.0.0.3,show ip route
3,C004,DHCP pool configured with wrong network,C017,0.42,Medium,Missing ip helper-address,show running-config
4,C005,R1 G0/0 administratively shut down,C007,0.48,Medium,R1 has incorrect static route next hop 10.0.0.3,show ip route
5,C006,PC1 and PC2 configured with the same IP addres...,C003,0.31,Low,VLAN 10 not allowed on trunk,show interfaces trunk
6,C007,R1 has incorrect static route next hop 10.0.0.3,C003,0.51,Medium,VLAN 10 not allowed on trunk,show interfaces trunk
7,C008,PC1 configured with incorrect DNS server 192.1...,C057,0.25,Low,Incorrect NTP server configuration,show ntp associations
8,C009,Ethernet cable between Switch1 Fa0/2 and PC2 F...,C003,0.44,Medium,VLAN 10 not allowed on trunk,show interfaces trunk
9,C010,Wrong Ethernet cable type used between PC1 Fa0...,C009,0.37,Low,Ethernet cable between Switch1 Fa0/2 and PC2 F...,show interfaces status


## 37. Compare AI Diagnosis with Expected Fault

The probable fault generated by the similarity-based AI-assisted system is compared with the expected fault available in the original dataset.

This comparison helps evaluate how often the retrieved similar case has the same fault category as the current case.

In [69]:
df["AI_Match"] = (
    df["Expected_Fault"].str.lower().str.strip() ==
    df["AI_Probable_Fault"].str.lower().str.strip()
)

print("AI diagnosis comparison completed.")

AI diagnosis comparison completed.


## 38. Calculate AI-Assisted Diagnosis Accuracy

The accuracy is calculated by comparing the AI-assisted probable fault with the expected fault.

The result represents the percentage of cases where the similarity-based diagnosis matches the expected fault.

In [70]:
correct_predictions = df["AI_Match"].sum()
total_cases = len(df)

accuracy = (correct_predictions / total_cases) * 100

print("Correct AI Diagnoses:", correct_predictions)
print("Total Cases:", total_cases)
print("AI-Assisted Diagnosis Accuracy:", round(accuracy, 2), "%")

Correct AI Diagnoses: 0
Total Cases: 100
AI-Assisted Diagnosis Accuracy: 0.0 %


## 39. Analyze Confidence Level Distribution

The confidence levels generated from similarity scores are analyzed to understand the reliability of the AI-assisted diagnosis system.

The confidence levels are categorized as High, Medium, or Low based on the similarity score.

In [71]:
confidence_count = df["AI_Confidence"].value_counts()

print("Confidence Level Distribution:")
print(confidence_count)

Confidence Level Distribution:
AI_Confidence
Low       72
Medium    28
Name: count, dtype: int64


## 40. Display High Confidence Diagnoses

High confidence cases have a strong similarity with an existing network fault case.

These cases can be considered more reliable for automated troubleshooting suggestions.

In [72]:
high_confidence = df[df["AI_Confidence"] == "High"]

print("Number of High Confidence Cases:", len(high_confidence))

high_confidence[[
    "Case_ID",
    "Expected_Fault",
    "AI_Probable_Fault",
    "AI_Similarity_Score"
]].head(10)

Number of High Confidence Cases: 0


,Case_ID,Expected_Fault,AI_Probable_Fault,AI_Similarity_Score


## 41. Display Low Confidence Diagnoses

Low confidence cases have limited similarity with existing network fault cases.

These cases may require additional troubleshooting information or human expert review.

In [73]:
low_confidence = df[df["AI_Confidence"] == "Low"]

print("Number of Low Confidence Cases:", len(low_confidence))

low_confidence[[
    "Case_ID",
    "Expected_Fault",
    "AI_Probable_Fault",
    "AI_Similarity_Score"
]].head(10)

Number of Low Confidence Cases: 72


,Case_ID,Expected_Fault,AI_Probable_Fault,AI_Similarity_Score
5,C006,PC1 and PC2 configured with the same IP addres...,VLAN 10 not allowed on trunk,0.31
7,C008,PC1 configured with incorrect DNS server 192.1...,Incorrect NTP server configuration,0.25
9,C010,Wrong Ethernet cable type used between PC1 Fa0...,Ethernet cable between Switch1 Fa0/2 and PC2 F...,0.37
10,C011,Incorrect static ARP entry for PC2,Ethernet cable between Switch1 Fa0/2 and PC2 F...,0.31
11,C012,HTTP service disabled on the server,HTTP application service is not running on the...,0.38
14,C015,Switch2 assigned to the wrong EtherChannel group,Uneven traffic is caused by normal load-balanc...,0.33
18,C019,Incorrect DTP negotiation,Required VLAN is excluded from the permitted t...,0.34
20,C021,VTP server/client mode mismatch,Incorrect VTP domain,0.40
22,C023,PortFast configured on wrong port,Incorrect STP root bridge,0.26
23,C024,BPDU Guard triggered,Wrong violation mode,0.24


## 42. Display AI Diagnosis Summary

This section provides a summary of the AI-assisted diagnosis results.

In [75]:
print("----------------------------------------")
print("NETFORENSICS AI DIAGNOSIS SUMMARY")
print("----------------------------------------")

print("Total Cases:", total_cases)
print("Correct Diagnoses:", correct_predictions)
print("AI Diagnosis Accuracy:", round(accuracy, 2), "%")

print("\nConfidence Distribution:")
print(df["AI_Confidence"].value_counts())



----------------------------------------
NETFORENSICS AI DIAGNOSIS SUMMARY
----------------------------------------
Total Cases: 100
Correct Diagnoses: 0
AI Diagnosis Accuracy: 0.0 %

Confidence Distribution:
AI_Confidence
Low       72
Medium    28
Name: count, dtype: int64


## 43. Save AI-Assisted Diagnosis Results

The dataset containing the AI-assisted diagnosis results is saved as a CSV file.

This file will be used in the next phases of the NetForensics AI project, including human review and final visualization.

In [76]:
df.to_csv("ai_assisted_diagnosis_results.csv", index=False)

print("AI-assisted diagnosis results saved successfully.")

AI-assisted diagnosis results saved successfully.


# 44. Conclusion

In this phase, an AI-assisted network fault diagnosis approach was developed for the NetForensics AI project.

Initially, pre-trained language models were tested for generating network fault diagnoses. However, the small language models produced incomplete or unreliable responses for structured network troubleshooting.

Therefore, a similarity-based AI-assisted approach was implemented using TF-IDF vectorization and cosine similarity.

The system compares the symptom and evidence of a network problem with historical troubleshooting cases and identifies the most similar case.

The following activities were completed:

- Created structured prompts for network fault diagnosis.
- Tested pre-trained language models for diagnosis.
- Evaluated the limitations of small language models.
- Combined symptom and evidence for text analysis.
- Applied TF-IDF vectorization.
- Calculated cosine similarity between network cases.
- Retrieved similar historical troubleshooting cases.
- Generated probable faults and troubleshooting recommendations.
- Calculated confidence levels based on similarity scores.
- Applied AI-assisted diagnosis to all network cases.
- Evaluated the generated diagnoses.
- Saved the AI-assisted diagnosis results.

The similarity-based approach provides an explainable and dataset-driven method for network fault diagnosis. Low-confidence cases can be forwarded for human expert review, while higher-confidence cases can provide faster troubleshooting recommendations.

The next phase of the NetForensics AI project will focus on human review and responsible AI practices.